# 1. 패키지 설치

In [1]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters langchain-pinecone docx2txt

Note: you may need to restart the kernel to use updated packages.


# 2. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [2]:
# from langchain_community.document_loaders import Docx2txtLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1500,
#     chunk_overlap=200,
# )

# loader = Docx2txtLoader('./KB_Direct_LngtrmDriver_202601.docx')
# document_list = loader.load_and_split(text_splitter=text_splitter)

## 2.1 pdf 처리를 위한 PyMuPDFLoader 사용

In [3]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# pdf1.pdf, pdf2.pdf를 'data/' 폴더에 넣기
loader = DirectoryLoader(
    "data/",  # 폴더 경로
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)
docs = loader.load()  # 모든 PDF 로드
print(f"로드된 문서 수: {len(docs)}")  # 2개 이상

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, chunk_overlap=200
    )
chunks = text_splitter.split_documents(docs)


c:\Users\dlcksgml\miniconda3\envs\langchain-insurance\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


로드된 문서 수: 1148


In [5]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 3. Pinecone 벡터데이터베이스에 저장

In [6]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'hanvely-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 데이터를 추가할 때
database = PineconeVectorStore.from_documents(
  chunks, 
  embedding, 
  index_name=index_name
  )

In [7]:
query = '과실비율을 확정할 때 따라야 할 기본 원칙 두 가지'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
retriever = database.as_retriever(search_kwargs={'k': 4})
retriever.invoke(query)

[Document(id='39b3f6e7-87cb-4739-bacc-f8b77676b317', metadata={'author': '', 'creationDate': "D:20251203131713+09'00'", 'creationdate': '2025-12-03T13:17:13+09:00', 'creator': 'Adobe InDesign 21.0 (Windows)', 'encryption': 'Standard V5 R6 256-bit AES', 'file_path': 'data\\230630_자동차사고 과실비율 인정기준_최종.pdf', 'format': 'PDF 1.7', 'keywords': '', 'modDate': "D:20251203132822+09'00'", 'moddate': '2025-12-03T13:28:22+09:00', 'page': 20.0, 'producer': 'Adobe PDF Library 18.0', 'source': 'data\\230630_자동차사고 과실비율 인정기준_최종.pdf', 'subject': '', 'title': '', 'total_pages': 600.0, 'trapped': ''}, page_content='자동차사고 과실비율 인정기준 │ 제2편 총설\n020\n4. 과실비율 인정기준의 적용\n(1) 기본 과실비율의 일반적인 설정방법\n통상 자동차 사고에 있어서 피해자 과실의 유무 및 그 정도는 사고 발생에 기여한 피해자의 \n교통법규 위반의 유무 및 그 정도의 형태로 구체화된다. 따라서, 과실비율을 결정함에 있어서는 \n먼저 교통법규와 이에 기한 통행우선권을 기본으로 하여 그 비율을 수치화한 다음, 구체적인 \n상황에서의 위험성에 관한 요소들, 즉 차량속도, 도로상황, 사고발생 지점의 도로 구조, 사고 \n차량간 거리, 기타 관련된 교통상황 등을 고려하여 사고 발생의 예견 내지 회피 가능한 요소들\n을 추출, 적용함으로써 기본 과실비율을 수정하는 과정을 거치게 된다.\n본서에서의 각 사고 유형별 기본 과실비

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [8]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

In [9]:
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [10]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [11]:
ai_message = qa_chain({"query": query})

C:\Users\dlcksgml\AppData\Local\Temp\ipykernel_11540\3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  ai_message = qa_chain({"query": query})


In [12]:
ai_message = qa_chain.invoke({"query": query})

In [13]:
ai_message

{'query': '과실비율을 확정할 때 따라야 할 기본 원칙 두 가지',
 'result': '과실비율을 확정할 때 따라야 할 기본 원칙 두 가지는 다음과 같습니다. 첫째, 수정요소가 사고의 발생 또는 손해의 확대에 영향을 끼친 것으로 판단될 때에만 가산하거나 감산합니다. 둘째, 기본 과실비율에서 지나치게 큰 폭의 수정요소 가산 또는 감산, 수정요소의 중복 적용은 양자의 입장을 좁히기 어려우며, 사고의 과실예측력을 저하하므로 객관적이지 않는 무조건적인 수정요소의 적용보다는 사고와 인과관계가 있는 주요 수정요인에 대하여 적용해야 합니다.'}